# 03 · Caracterización del alfa y evolución temporal

**Proyecto final · Reactividad alfa en EEG durante la alternancia de ojos abiertos y cerrados**

| | |
|---|---|
| Dueño | C · Diego |
| Etapas de la rúbrica | Etapas 3 y 4 (20 %) |
| Entregable técnico | `potencia_alfa()`, `resumen_por_bloque()` |
| Figuras | fig03_psd_eo_vs_ec · fig04_alfa_en_el_tiempo · fig05_comparacion_bloques |

Reglas: este notebook solo lo edita su dueño. Se abre desde GitHub, se guarda con *Archivo → Guardar una copia en GitHub* en tu rama
(nunca en `main`) y antes de guardar se borran las salidas (*Editar → Borrar todas las salidas*). Las figuras se guardan con
`alfa.guardar_figura(fig, "figNN_nombre")` en la carpeta `figuras/` de Drive.

## 0. Preparación

In [ ]:
# ── Preparación (igual en todos los notebooks; no editar) ──────────────
RAMA = "main"    # cámbiala solo si necesitas un alfa.py que aún está en un PR

import os, sys
EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO = "/content/repo"
    URL = "https://github.com/moiseiseiseis/Alpha-reactivity-during-resting-state-EEG.git"
    if not os.path.isdir(REPO):
        !git clone -q -b {RAMA} {URL} {REPO}
    else:
        !git -C {REPO} fetch -q origin {RAMA} && git -C {REPO} checkout -q {RAMA} && git -C {REPO} reset -q --hard origin/{RAMA}
    %pip install -q mne
    sys.path.insert(0, f"{REPO}/src")
else:
    sys.path.insert(0, os.path.abspath("../src"))

import importlib
import alfa
importlib.reload(alfa)   # recoge cambios de alfa.py sin reiniciar el entorno

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne

R = alfa.rutas()
print("Datos en:", R["datos"])
print("Registro encontrado:", R["vhdr"].exists())

In [ ]:
# Parámetros en uso: todo sale de CONFIG (no escribir números a mano en el notebook)
cfg = alfa.CONFIG
alfa.imprimir_config(cfg)

In [ ]:
raw = alfa.cargar_raw(R["vhdr"])
raw = alfa.marcar_artefactos(raw, cfg)    # anotaciones BAD_ (amplitud, canal plano, parpadeos, manuales)
bloques = alfa.tabla_bloques(raw)
raw_ac = alfa.acondicionar(raw, cfg)      # conserva las anotaciones BAD_ al filtrar y remuestrear

print(raw_ac)
display(bloques)
display(alfa.resumen_artefactos(raw, bloques))   # cuánto se marcó en cada bloque


## 1. Contenido frecuencial por condición

- [ ] PSD de Welch promedio EO vs EC en los canales elegidos
- [ ] Pico alfa individual (frecuencia) en EC

In [ ]:
# TODO

# ── PSD de Welch por bloque (canales de CONFIG) ──────────────────────────
canales = cfg["canales"]
sf = raw_ac.info["sfreq"]
n_fft = int(cfg["ventana_s"] * sf)                  # 2 s → 500 muestras → resolución de 0.5 Hz
n_overlap = int(n_fft * cfg["solapamiento"])         # 50 %

registros = []
registros = []
for _, b in bloques.iterrows():
    tmin = b["inicio_s"] + cfg["recorte_transicion_s"]   # descarta los primeros 5 s tras el cambio
    tramo = raw_ac.copy().crop(tmin=tmin, tmax=b["fin_s"])
    epocas = mne.make_fixed_length_epochs(
        tramo,
        duration=cfg["ventana_s"],
        overlap=cfg["ventana_s"] * cfg["solapamiento"],
        reject_by_annotation=cfg["excluir_artefactos"],    # descarta épocas que tocan un BAD_
        preload=True, verbose=False,
    ).pick(canales)
    if len(epocas) == 0:
        print(f"Bloque {b['bloque']}: sin épocas limpias, se omite")
        continue
    spec = epocas.compute_psd(method="welch", fmin=1, fmax=40, n_fft=n_fft, verbose=False)
    registros.append({
        "bloque": b["bloque"], "condicion": b["condicion"],
        "n_epocas": len(epocas),
        "psd": spec.get_data().mean(axis=0) * 1e12,       # promedio de épocas → (canales, frecuencias), µV²/Hz
    })
freqs = spec.freqs

print("Épocas limpias por bloque:", {r["bloque"]: r["n_epocas"] for r in registros})

# Promedio por condición: primero entre canales, luego entre bloques
psd_bloque = {c: np.array([r["psd"].mean(axis=0) for r in registros if r["condicion"] == c])
              for c in ["EO", "EC"]}                 # cada uno: (8 bloques, frecuencias)
psd_media = {c: v.mean(axis=0) for c, v in psd_bloque.items()}
psd_sem = {c: v.std(axis=0, ddof=1) / np.sqrt(len(v)) for c, v in psd_bloque.items()}

# ── Pico alfa individual (IAF) en 7–14 Hz ─────────────────────────────────
rango = (freqs >= 7) & (freqs <= 14)
for c in ["EC", "EO"]:
    i = np.argmax(psd_media[c][rango])
    print(f"{c}: pico en {freqs[rango][i]:.1f} Hz ({psd_media[c][rango][i]:.2f} µV²/Hz)")
iaf = freqs[rango][np.argmax(psd_media["EC"][rango])]

# Pico por bloque EC, para ver si es estable a lo largo del registro
iaf_bloques = [freqs[rango][np.argmax(v[rango])] for v in psd_bloque["EC"]]
print("IAF por bloque EC:", iaf_bloques)
print(f"Resolución en frecuencia: {freqs[1] - freqs[0]:.2f} Hz")

## 2. Medida de actividad alfa

- [ ] Potencia absoluta 8–13 Hz (µV²) y relativa (fracción de la potencia total)
- [ ] Justificar cuál se usa como medida principal

In [ ]:
df = alfa.potencia_alfa(raw_ac, bloques, cfg)

print(df.shape, "| ventanas por canal:", df["t_s"].nunique())
print(df["dentro"].value_counts().rename("ventanas dentro de bloque (sin recorte)"))
display(df.head())
display(df[df["dentro"]].groupby("condicion")[["alfa_abs", "alfa_rel"]].median().round(3))

### Medida principal

- **Absoluta (µV²):** Es la medida directa y la más común para la reactividad alfa. Un punto "en contra" a considerar es que cambia también con cosas que no son alfa, como la impedancia de los electrodos o los artefactos de banda ancha.
- **Relativa:** Normaliza cada ventana, así que se cancelan los cambios de ganancia globales. Pero depende de las otras bandas. Si en EO aumentan theta, beta o actividad muscular, el alfa relativo baja aunque el alfa en sí no haya cambiado.
- **Una salida razonable:** La absoluta como principal y la relativa como verificación. Si las dos dan la misma conclusión (EC > EO en todos los bloques), el resultado no depende de la normalización. Esa es justo la clase de evidencia que _Ascencio_ puede usar en robustez.

## 3. Figura 3 · PSD EO vs EC

- [ ] Banda 8–13 Hz marcada, escala log en potencia

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
colores = {"EO": "tab:blue", "EC": "tab:red"}
etiquetas = {"EO": "Ojos abiertos (EO)", "EC": "Ojos cerrados (EC)"}

for c in ["EO", "EC"]:
    ax.semilogy(freqs, psd_media[c], color=colores[c], lw=2, label=etiquetas[c])
    ax.fill_between(freqs, psd_media[c] - psd_sem[c], psd_media[c] + psd_sem[c],
                    color=colores[c], alpha=0.2)

a, b_ = cfg["banda_alfa"]
ax.axvspan(a, b_, color="gold", alpha=0.25, label=f"Banda alfa {a:g}–{b_:g} Hz")
ax.axvline(iaf, color="k", ls="--", lw=1, label=f"Pico alfa EC = {iaf:.1f} Hz")

ax.set_xlim(1, 40)
ax.set_xlabel("Frecuencia (Hz)")
ax.set_ylabel("Densidad espectral de potencia (µV²/Hz, escala log)")
ax.set_title(f"PSD de Welch por condición · promedio de {len(canales)} canales posteriores · "
             f"media ± EEM de 8 bloques")
ax.legend()
ax.grid(alpha=0.3, which="both")
plt.tight_layout()
plt.show()

alfa.guardar_figura(fig, "fig03_psd_eo_vs_ec")

## 4. Comparación cuantitativa EO vs EC

- [ ] Valores por bloque (no solo espectros)
- [ ] Razón EC/EO por par de bloques

In [ ]:
resumen = alfa.resumen_por_bloque(df)
resumen.to_csv(R["resultados"] / "resumen_por_bloque.csv", index=False)
display(resumen.round(3))

razones = resumen.drop_duplicates("par").set_index("par")[["razon_ec_eo_abs", "razon_ec_eo_rel"]]
print("\nRazón EC/EO por par de bloques:")
display(razones.round(2))
print(razones.agg(["mean", "std", "min", "max"]).round(2))
print(f"\nPares con EC > EO: abs {(razones['razon_ec_eo_abs'] > 1).sum()}/8 · "
      f"rel {(razones['razon_ec_eo_rel'] > 1).sum()}/8")

####El aumento del alfa al cerrar los ojos aparece en cada bloque del registro, no solo en el promedio.

In [ ]:
from scipy.stats import spearmanr

ec = resumen[resumen["condicion"] == "EC"]
eo = resumen[resumen["condicion"] == "EO"]
razones_par = resumen.drop_duplicates("par")

for nombre, x, y in [("Alfa abs EC", ec["bloque"], ec["alfa_abs_media"]),
                     ("Alfa abs EO", eo["bloque"], eo["alfa_abs_media"]),
                     ("Razón EC/EO abs", razones_par["par"], razones_par["razon_ec_eo_abs"]),
                     ("Razón EC/EO rel", razones_par["par"], razones_par["razon_ec_eo_rel"])]:
    rho, p = spearmanr(x, y)
    print(f"{nombre:<16} ρ de Spearman = {rho:+.2f} (p = {p:.3f}, n = {len(x)})")

- El alfa en EC crece de forma casi monótona (ρ = +0.93, p = 0.001); la excepción es el bloque 9, cuyo alfa en EC es menor que el del bloque anterior. La razón EC/EO absoluta también aumenta (ρ = +0.81, p = 0.015), pero la relativa ya no es significativa (ρ = +0.57, p = 0.14).
- Como EO no tiene tendencia, el aumento de la reactividad viene de que el alfa con ojos cerrados crece, no de que el alfa con ojos abiertos baje.  Esto ayuda a descartar una deriva global de amplitud (impedancia o ganancia), que habría movido las dos condiciones.
- Limitaciones:
  - Con n = 8 bloques, los valores de p son orientativos.
  - El tiempo y el orden de los bloques no se pueden separar: el EC más tardío siempre es el último. Con este diseño no se puede saber si el efecto es por el tiempo, por la fatiga o por la repetición.
  - Es un solo participante: el hallazgo describe este registro y no se puede generalizar

## 5. Figura 4 · Alfa a lo largo del registro

- [ ] Potencia por ventana vs tiempo, bloques sombreados

In [ ]:
# Promedio entre canales por ventana (todas las ventanas, también transiciones)
serie = df.groupby("t_s", as_index=False)[["alfa_abs", "alfa_rel"]].mean()

fig, axs = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
color_cond = {"EO": "tab:blue", "EC": "tab:red"}

for ax, col, etiqueta in [(axs[0], "alfa_abs", "Potencia alfa absoluta (µV², escala log)"),
                          (axs[1], "alfa_rel", "Potencia alfa relativa (fracción)")]:
    ax.plot(serie["t_s"], serie[col], color="0.25", lw=0.8)
    for _, b in bloques.iterrows():
        ax.axvspan(b["inicio_s"], b["fin_s"], color=color_cond[b["condicion"]], alpha=0.12)
        ax.axvspan(b["inicio_s"], b["inicio_s"] + cfg["recorte_transicion_s"], color="0.6", alpha=0.3)
    for _, r in resumen.iterrows():                       # media del bloque (ventanas 'dentro')
        b = bloques.loc[bloques["bloque"] == r["bloque"]].iloc[0]
        ax.hlines(r[f"{col}_media"], b["inicio_s"] + cfg["recorte_transicion_s"], b["fin_s"],
                  color=color_cond[r["condicion"]], lw=2.5)
    ax.set_ylabel(etiqueta)
    ax.grid(alpha=0.3, which="both")
axs[0].set_yscale("log")

from matplotlib.patches import Patch
from matplotlib.lines import Line2D
axs[0].legend(handles=[Patch(color="tab:red", alpha=0.3, label="Ojos cerrados (EC)"),
                       Patch(color="tab:blue", alpha=0.3, label="Ojos abiertos (EO)"),
                       Patch(color="0.6", alpha=0.5, label=f"Recorte de {cfg['recorte_transicion_s']:g} s"),
                       Line2D([], [], color="k", lw=2.5, label="Media del bloque")],
              loc="upper left", ncol=4, fontsize=9)
axs[1].set_xlabel("Tiempo (s)")
axs[0].set_title(f"Potencia alfa ({cfg['banda_alfa'][0]:g}–{cfg['banda_alfa'][1]:g} Hz) por ventana de "
                 f"{cfg['ventana_s']:g} s · promedio de {len(cfg['canales'])} canales posteriores")
plt.tight_layout()
plt.show()

alfa.guardar_figura(fig, "fig04_alfa_en_el_tiempo")

## 6. Transiciones

- [ ] Curva promedio alrededor de cada cambio EO→EC y EC→EO
- [ ] ¿El cambio aparece igual después de cada transición? ¿Cuánto tarda?

In [ ]:
# TODO
# ── Curva promedio alrededor de cada transición ──────────────────────────
pre, post = 15, 20
grid = np.arange(-pre, post + 1, 1.0)           # resolución de 1 s (una ventana por segundo)

def tramo(t0, col):
    y = np.interp(t0 + grid, serie["t_s"], serie[col], left=np.nan, right=np.nan)
    return np.log10(y) if col == "alfa_abs" else y   # la absoluta se promedia en log

transiciones = {
    "EO→EC (cerrar)": bloques[(bloques["condicion"] == "EC") & (bloques["bloque"] > 1)],
    "EC→EO (abrir)":  bloques[bloques["condicion"] == "EO"],
}
color_t = {"EO→EC (cerrar)": "tab:red", "EC→EO (abrir)": "tab:blue"}

fig, axs = plt.subplots(1, 2, figsize=(14, 4.5))
latencias = []
for ax, col, etiqueta in [(axs[0], "alfa_abs", "Potencia alfa absoluta (µV², escala log)"),
                          (axs[1], "alfa_rel", "Potencia alfa relativa (fracción)")]:
    for nombre, bl in transiciones.items():
        M = np.array([tramo(t0, col) for t0 in bl["inicio_s"]])          # (transiciones, tiempo)
        media = np.nanmean(M, axis=0)
        eem = np.nanstd(M, axis=0, ddof=1) / np.sqrt(len(M))
        f = (lambda v: 10**v) if col == "alfa_abs" else (lambda v: v)
        ax.plot(grid, f(media), color=color_t[nombre], lw=2, label=f"{nombre} · n = {len(M)}")
        ax.fill_between(grid, f(media - eem), f(media + eem), color=color_t[nombre], alpha=0.2)

        if col == "alfa_abs":                                             # latencia por transición
            for t0, curva in zip(bl["inicio_s"], M):
                antes = np.nanmean(curva[(grid >= -10) & (grid <= -2)])
                despues = np.nanmean(curva[(grid >= 6) & (grid <= 15)])
                medio = (antes + despues) / 2
                cruza = (curva >= medio) if despues > antes else (curva <= medio)
                idx = np.where(cruza & (grid >= -2))[0]
                latencias.append({"transicion": nombre, "inicio_s": round(t0, 1),
                                  "cambio_x": round(10**(despues - antes), 2),
                                  "latencia_s": grid[idx[0]] if len(idx) else np.nan})
    ax.axvline(0, color="k", lw=1)
    ax.axvspan(0, cfg["recorte_transicion_s"], color="0.6", alpha=0.3)
    ax.set_xlabel("Tiempo desde el inicio del nuevo bloque (s)")
    ax.set_ylabel(etiqueta)
    ax.grid(alpha=0.3, which="both")
axs[0].set_yscale("log")
axs[0].legend()
fig.suptitle("Potencia alfa alrededor de las transiciones · media ± EEM entre transiciones")
plt.tight_layout()
plt.show()
alfa.guardar_figura(fig, "fig04b_transiciones")

lat = pd.DataFrame(latencias)
display(lat)
print(lat.groupby("transicion")[["cambio_x", "latencia_s"]].agg(["median", "min", "max"]))

Al abrir los ojos el EC se vuelve EO; mientras que al cerrar los ojos el EO se vuelve EC.  
Hay una asimetría: abrir los ojos apaga el alfa rápido y por completo. Cerrarlos lo recupera de golpe por un momento, pero el nivel estable tarda más de 15 s en alcanzarse

## 7. Figura 5 · Comparación entre bloques

- [ ] Boxplot o puntos por bloque, EO y EC lado a lado
- [ ] Bloques que se apartan del comportamiento general y posibles causas

In [ ]:
# alfa.guardar_figura(fig, "fig05_comparacion_bloques")
d = df[df["dentro"]].groupby(["bloque", "condicion", "t_s"], as_index=False)["alfa_abs"].mean()
d["par"] = (d["bloque"] + 1) // 2

fig, ax = plt.subplots(figsize=(13, 5))
desplaz = {"EC": -0.2, "EO": 0.2}
for c in ["EC", "EO"]:
    datos = [d[(d["par"] == p) & (d["condicion"] == c)]["alfa_abs"].values for p in range(1, 9)]
    pos = np.arange(1, 9) + desplaz[c]
    bp = ax.boxplot(datos, positions=pos, widths=0.35, patch_artist=True, showfliers=False)
    for caja in bp["boxes"]:
        caja.set(facecolor=color_cond[c], alpha=0.35)
    for m in bp["medians"]:
        m.set(color=color_cond[c], lw=2)
    medias = resumen[resumen["condicion"] == c].sort_values("par")["alfa_abs_media"].values
    ax.plot(pos, medias, "o", color=color_cond[c], mec="k", label=f"{c} · media del bloque")

ax.set_yscale("log")
ax.set_ylim(top=ax.get_ylim()[1] * 2)          # deja espacio arriba para las etiquetas
for p, r in zip(razones_par["par"], razones_par["razon_ec_eo_abs"]):
    ax.text(p, 0.97, f"EC/EO ×{r:.1f}", transform=ax.get_xaxis_transform(),
            ha="center", va="top", fontsize=10, fontweight="bold")

ax.set_xticks(range(1, 9))
ax.set_xticklabels([f"Par {p}\n(bloques {2*p-1}–{2*p})" for p in range(1, 9)])
ax.set_ylabel("Potencia alfa absoluta (µV², escala log)")
ax.set_title("Potencia alfa por bloque (ventanas de 2 s tras el recorte)\n"
             "Caja = distribución de ventanas · punto = media del bloque · EC/EO ×N = razón del par")
ax.legend(loc="lower right")
ax.grid(alpha=0.3, which="both", axis="y")
plt.tight_layout()
plt.show()
alfa.guardar_figura(fig, "fig05_comparacion_bloques")

## Interpretación breve

**Observaciones**

- **Reactividad alfa clara y consistente.** Con ojos cerrados aparece un pico espectral en **10.5 Hz** (pico alfa individual), ~6.6 veces más alto que con ojos abiertos. La potencia absoluta en 8–13 Hz es mayor en EC que en EO en los **8 de 8 pares** de bloques, con una razón EC/EO media de **4.2** (rango 2.2–6.2). La potencia relativa da la misma conclusión (razón media 2.2, rango 1.7–2.5).
- **El alfa en EC aumenta a lo largo del registro.** La potencia alfa en EC crece casi de forma monótona entre bloques (ρ de Spearman = +0.93, p = 0.001, n = 8), mientras que en EO se mantiene estable (ρ = +0.24, p = 0.57). La razón EC/EO absoluta también aumenta (ρ = +0.81, p = 0.015); la relativa muestra la misma dirección, pero de forma más débil y no significativa (ρ = +0.57, p = 0.14). El bloque 9 es la excepción: su alfa en EC es menor que el del bloque anterior.
- **Dinámica asimétrica en las transiciones.** Al abrir los ojos, el alfa cae a ~1/5 de su nivel en pocos segundos. Al cerrarlos aparece un aumento breve e intenso en el primer segundo, seguido de una caída y de una recuperación gradual que no se estabiliza antes de ~15–20 s. El cambio ocurre dentro de ±2 s del marcador de bloque.
- **Variabilidad dentro de los bloques.** En EC, la potencia alfa fluctúa mucho entre ventanas (aparece en ráfagas); a nivel de ventanas individuales, las distribuciones de EC y EO se traslapan, aunque las medias por bloque se separan claramente.
- **Efecto de la exclusión de artefactos.** Se excluyó el 6.3 % del tiempo en EO (sobre todo parpadeos) y el 0.4 % en EC. Al excluir los parpadeos se redujo el exceso de potencia de EO por debajo de ~6 Hz, lo que indica que ese exceso provenía en gran parte de actividad ocular.

**Interpretación**

El aumento de la potencia alfa al cerrar los ojos en regiones occipitales es consistente con el efecto de Berger descrito en la literatura. El crecimiento progresivo del alfa en EC podría relacionarse con cambios en el estado de relajación o de vigilancia durante el registro, pero los datos no permiten identificar la causa. Como la potencia en EO no cambia, es poco probable que el aumento se deba a una deriva global de la amplitud de la señal.

**Limitaciones**

Se trata de un solo participante y de 8 pares de bloques, por lo que los valores de p son orientativos y los resultados no se pueden generalizar. El orden temporal de los bloques no se puede separar de efectos de repetición o fatiga. Las latencias de las transiciones tienen una resolución de ~1–2 s, el marcador del bloque no coincide necesariamente con el momento real de abrir o cerrar los ojos, y la exclusión de parpadeos (frecuentes justo al abrirlos) deja huecos en la serie temporal. La lenta recuperación del alfa tras cerrar los ojos indica que el recorte de 5 s no captura el estado estable, por lo que las medias de EC dependen del recorte elegido (se evalúa en la etapa de robustez).